# CSCI E-89 Assignment 03 — Problem 4

**Student:** Levente Papp

## Fashion-MNIST Image Classifier with PyTorch

This notebook follows the *Building an Image Classifier with PyTorch* workflow from the course notebook. It loads and explores Fashion-MNIST, defines and trains a multilayer perceptron, compares training and validation accuracy, and evaluates predictions on unseen data.


## 1. Setup

Import the required libraries, make the experiment reproducible, and select the best available PyTorch device in the order CUDA, MPS, then CPU.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import torchvision
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader, random_split

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"PyTorch {torch.__version__}, torchvision {torchvision.__version__}, torchmetrics {torchmetrics.__version__}")
print(f"NumPy {np.__version__}, Matplotlib {plt.matplotlib.__version__}")
print(f"Random seed: {SEED}")
print(f"Using device: {device}")

## 2. Load and Prepare Fashion-MNIST

Convert each image to a `float32` tensor scaled to `[0, 1]`. The original 60,000-image training set is reproducibly divided into 55,000 training images and 5,000 validation images. All loaders use batches of 32.


In [ ]:
BATCH_SIZE = 32
DATA_DIR = Path("prob4/datasets")

transform = T.Compose([
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
])

train_and_valid_data = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=transform
)
test_data = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=transform
)

split_generator = torch.Generator().manual_seed(SEED)
train_data, valid_data = random_split(
    train_and_valid_data, [55_000, 5_000], generator=split_generator
)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_data, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator
)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

print(f"Training set:   {len(train_data):,} images")
print(f"Validation set: {len(valid_data):,} images")
print(f"Test set:       {len(test_data):,} images")
images, labels = next(iter(train_loader))
print(f"Image batch shape: {tuple(images.shape)}")
print(f"Label batch shape: {tuple(labels.shape)}")

## 3. Inspect Training Samples

The labels are class indices from 0 through 9. The dataset's `classes` attribute maps them to readable clothing categories.


In [ ]:
N_ROWS = 4
N_COLUMNS = 8
class_names = train_and_valid_data.classes

figure, axes = plt.subplots(N_ROWS, N_COLUMNS, figsize=(12, 7.2))
for index, axis in enumerate(axes.flat):
    image, label = train_data[index]
    axis.imshow(image.squeeze(0), cmap="binary")
    axis.set_title(class_names[label], fontsize=9)
    axis.axis("off")

figure.suptitle("Fashion-MNIST sample training images")
figure.tight_layout()
figure_path = Path("prob4/figures/samples.png")
figure_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(figure_path, dpi=100, bbox_inches="tight")
plt.show()

## 4. Evaluation and Training Helpers

`evaluate` accumulates a TorchMetrics metric over an entire data loader. `train` performs optimization and records the mean training loss plus training and validation accuracy after every epoch.


In [ ]:
def evaluate(model, data_loader, metric):
    """Compute a TorchMetrics metric over every batch in a data loader."""
    model_device = next(model.parameters()).device
    metric = metric.to(model_device)
    model.eval()
    metric.reset()
    with torch.no_grad():
        for images, labels in data_loader:
            logits = model(images.to(model_device))
            metric.update(logits, labels.to(model_device))
    return metric.compute()


def train(model, optimizer, loss_function, accuracy_metric,
          train_loader, valid_loader, n_epochs):
    """Train a classifier and return its per-epoch learning history."""
    model_device = next(model.parameters()).device
    accuracy_metric = accuracy_metric.to(model_device)
    history = {"train_loss": [], "train_accuracy": [], "valid_accuracy": []}

    for epoch in range(n_epochs):
        model.train()
        accuracy_metric.reset()
        total_loss = 0.0
        sample_count = 0

        for images, labels in train_loader:
            images, labels = images.to(model_device), labels.to(model_device)
            optimizer.zero_grad()
            logits = model(images)
            loss = loss_function(logits, labels)
            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            sample_count += batch_size
            accuracy_metric.update(logits.detach(), labels)

        train_loss = total_loss / sample_count
        train_accuracy = accuracy_metric.compute().item()
        valid_accuracy = evaluate(model, valid_loader, accuracy_metric).item()
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["valid_accuracy"].append(valid_accuracy)

        print(f"Epoch {epoch + 1:2d}/{n_epochs}: loss={train_loss:.4f}, "
              f"train_accuracy={train_accuracy:.4f}, valid_accuracy={valid_accuracy:.4f}")

    return history

## 5. Define the Neural Network

The classifier flattens each 28 × 28 image and passes it through hidden layers of 300 and 100 ReLU units. Its 10 outputs are logits, one per Fashion-MNIST class. Cross-entropy loss and stochastic gradient descent are used for training.


In [ ]:
class ImageClassifier(nn.Module):
    """Fully connected classifier that returns logits for ten classes."""

    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 300),
            nn.ReLU(),
            nn.Linear(300, 100),
            nn.ReLU(),
            nn.Linear(100, 10),
        )

    def forward(self, images):
        return self.network(images)


torch.manual_seed(SEED)
model = ImageClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
accuracy_metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(model)
print(f"Number of parameters: {parameter_count:,}")

## 6. Train for 20 Epochs

Train the model while retaining all measurements in `history` for later visualization.


In [ ]:
n_epochs = 20
history = train(
    model,
    optimizer,
    loss_function,
    accuracy_metric,
    train_loader,
    valid_loader,
    n_epochs,
)
print(f"Training complete; history contains {len(history['train_loss'])} epochs")

## 7. Compare Training and Validation Accuracy

Plot both accuracy series on the same axes to examine learning progress and the gap between performance on training and held-out data.


In [ ]:
epochs = range(1, len(history["train_accuracy"]) + 1)
figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(epochs, history["train_accuracy"], marker="o", linestyle="--",
          label="Training accuracy")
axis.plot(epochs, history["valid_accuracy"], marker="s", linestyle="-",
          label="Validation accuracy")
axis.set_xlabel("Epoch")
axis.set_ylabel("Accuracy")
axis.set_title("Fashion-MNIST Training and Validation Accuracy")
axis.set_xticks(list(epochs))
axis.grid(alpha=0.3)
axis.legend()
figure.tight_layout()

figure_path = Path("prob4/figures/accuracy.png")
figure_path.parent.mkdir(parents=True, exist_ok=True)
figure.savefig(figure_path, dpi=120, bbox_inches="tight")
plt.show()

## 8. Test Performance and Example Predictions

Finally, evaluate accuracy on the untouched test set. For three validation examples, report the predicted and true classes, all class probabilities, and the four most likely predictions.


In [ ]:
test_accuracy = evaluate(model, test_loader, accuracy_metric).item()
print(f"Test-set accuracy: {test_accuracy:.4f}")

model.eval()
images, true_labels = next(iter(valid_loader))
images = images[:3].to(next(model.parameters()).device)
true_labels = true_labels[:3]

with torch.no_grad():
    probabilities = F.softmax(model(images), dim=1).cpu()

predicted_labels = probabilities.argmax(dim=1)
top_probabilities, top_indices = probabilities.topk(k=4, dim=1)

print("\nPredictions for 3 validation images:")
for index in range(3):
    print(f"\nImage {index + 1}")
    print(f"  Predicted class: {class_names[predicted_labels[index].item()]}")
    print(f"  True class:      {class_names[true_labels[index].item()]}")
    class_probabilities = ", ".join(
        f"{name}={probability.item():.4f}"
        for name, probability in zip(class_names, probabilities[index])
    )
    print(f"  Class probabilities: {class_probabilities}")
    top_four = ", ".join(
        f"{class_names[class_index.item()]} ({probability.item():.4f})"
        for probability, class_index in zip(top_probabilities[index], top_indices[index])
    )
    print(f"  Top-4 predictions: {top_four}")